In [1]:
import polars as pl

arquivo_info = 'Music Info.csv'
arquivo_user = 'User Listening History.csv'

In [2]:
try:
    df_musica = pl.read_csv(arquivo_info)
    df_execucao = pl.read_csv(arquivo_user)

    # print(df_info.glimpse())
    # print(df_user.glimpse())
    df_execucao_agrupado = df_execucao.group_by('track_id').agg(
        pl.col('playcount').sum().alias('plays')
    )
    print(df_execucao_agrupado.head())

    df_merge = df_musica.join(
        df_execucao_agrupado, 
        left_on='track_id',
        right_on='track_id', 
        how='left'
    )
    # print(df_merge.head())
    print(df_merge.columns)

except Exception as e:
    print(f'Erro ao ler arquivos: {e}')

shape: (5, 2)
┌────────────────────┬───────┐
│ track_id           ┆ plays │
│ ---                ┆ ---   │
│ str                ┆ i64   │
╞════════════════════╪═══════╡
│ TRPDOIW128F1467354 ┆ 10    │
│ TRQDCRF12903D0311C ┆ 6     │
│ TRTBIEH128F427C13C ┆ 56    │
│ TRMQQZF128F932EEA5 ┆ 1277  │
│ TRDTODK128F1458778 ┆ 1     │
└────────────────────┴───────┘
['track_id', 'name', 'artist', 'spotify_preview_url', 'spotify_id', 'tags', 'genre', 'year', 'duration_ms', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'plays']


In [3]:
# Top 10 de músicas mais tocadas

top10_musicas = df_merge.group_by(['name', 'artist']).agg(
    pl.col('plays').sum()
).sort(by='plays', descending=True)
top10_musicas.head(10)

name,artist,plays
str,str,i64
"""Revelry""","""Kings of Leon""",527893
"""Alejandro""","""Lady Gaga""",111615
"""Gears""","""Miss May I""",111596
"""Halo""","""Depeche Mode""",91461
"""Bring Me To Life""","""Katherine Jenkins""",91448
"""Heartbreak Warfare""","""John Mayer""",87745
"""Uprising""","""Sabaton""",87050
"""Float On""","""Modest Mouse""",85079
"""Party In The U.S.A.""","""The Barden Bellas""",78443


In [ ]:
# Quantas musicas não foram tocadas?
df_sem_execucao = df_merge.group_by('plays').agg(
    pl.len()
).sort(by='plays', descending=False)
print(df_sem_execucao.head(1))

sem_execucao = df_merge.filter(
    pl.col('plays').is_null()
).height
print(sem_execucao)



shape: (1, 2)
┌───────┬───────┐
│ plays ┆ len   │
│ ---   ┆ ---   │
│ i64   ┆ u32   │
╞═══════╪═══════╡
│ null  ┆ 20224 │
└───────┴───────┘
20224


In [8]:
# Qual genero tem mais musicas não tocadas?
nao_tocadas = df_merge.filter(
    pl.col('plays').is_null()
).group_by('genre').agg(
    pl.len().alias('as_piores')
).sort('as_piores', descending=True)
print(nao_tocadas)

shape: (16, 2)
┌────────────┬───────────┐
│ genre      ┆ as_piores │
│ ---        ┆ ---       │
│ str        ┆ u32       │
╞════════════╪═══════════╡
│ null       ┆ 14329     │
│ Rock       ┆ 2386      │
│ Electronic ┆ 899       │
│ Metal      ┆ 450       │
│ Pop        ┆ 372       │
│ …          ┆ …         │
│ Folk       ┆ 91        │
│ Punk       ┆ 86        │
│ New Age    ┆ 55        │
│ Latin      ┆ 42        │
│ World      ┆ 22        │
└────────────┴───────────┘
